# ITELv5 · Notebook 08 · Exportación del Modelo Final
Entrena el meta-modelo (LogReg stacking) y exporta todos los artefactos.

**Flujo:** Embeddings → Split 80/10/10 → 5 MLP base → Stacking matrix → LogReg meta-modelo

In [11]:
import sys, os, warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
warnings.filterwarnings('ignore')

_anchor = 'ITELv5_01_CONFIGURACION.ipynb'
_cwd = Path('.').resolve()
if   (_cwd / _anchor).exists():                          BASE_DIR = _cwd.parent
elif (_cwd / 'notebooks' / _anchor).exists():            BASE_DIR = _cwd
elif (_cwd / 'research' / 'notebooks' / _anchor).exists(): BASE_DIR = _cwd / 'research'
else:                                                     BASE_DIR = _cwd.parent

class ITELConfig:
    BASE_DIR       = BASE_DIR
    DATA_DIR       = BASE_DIR / 'Datos_Brutos'
    PROCESADOS_DIR = BASE_DIR / 'processed_data'
    MODELS_DIR     = BASE_DIR.parent / 'assets' / 'models'
    SEED           = 42
    EMB_DIM        = 6144

ITELConfig.MODELS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PROCESADOS_DIR: {ITELConfig.PROCESADOS_DIR}')
print(f'MODELS_DIR    : {ITELConfig.MODELS_DIR}')


Device: cpu
PROCESADOS_DIR: C:\Users\USER\Documents\GitHub\ITEL\research\processed_data
MODELS_DIR    : C:\Users\USER\Documents\GitHub\ITEL\assets\models


In [12]:
class MLP(nn.Module):
    def __init__(self, in_dim=6144, hidden_dims=[512,256,128],
                 num_classes=2, dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

print('MLP definido  (in_dim=6144)')


MLP definido  (in_dim=6144)


In [13]:
# Cargar embeddings (.npy) y etiquetas (.csv)
npy_path = ITELConfig.PROCESADOS_DIR / 'embeddings_v5.npy'
csv_path = ITELConfig.PROCESADOS_DIR / 'embeddings_info_v5.csv'

if not npy_path.exists():
    raise FileNotFoundError(f'No se encontro {npy_path} - ejecuta primero NB05')

X_full  = np.load(npy_path).astype(np.float32)
df_info = pd.read_csv(csv_path)

# Target = categoria (Healthy / Patients)
y_raw  = df_info['categoria'].values
le     = LabelEncoder()
y_full = le.fit_transform(y_raw)

print(f'X shape : {X_full.shape}  (esperado N x 6144)')
print(f'Clases  : {le.classes_}  -> {np.bincount(y_full)}')


X shape : (3730, 6144)  (esperado N x 6144)
Clases  : ['Healthy' 'Patients']  -> [1680 2050]


In [14]:
# Split 80% train | 10% meta | 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full, y_full, test_size=0.20,
    random_state=ITELConfig.SEED, stratify=y_full
)
X_meta, X_test, y_meta, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    random_state=ITELConfig.SEED, stratify=y_temp
)

scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_meta_s  = scaler.transform(X_meta)
X_test_s  = scaler.transform(X_test)

scaler_path = ITELConfig.MODELS_DIR / 'final_mlp_scaler.joblib'
joblib.dump(scaler, scaler_path)
print(f'Train: {X_train_s.shape} | Meta: {X_meta_s.shape} | Test: {X_test_s.shape}')
print(f'Scaler guardado: {scaler_path}')


Train: (2984, 6144) | Meta: (373, 6144) | Test: (373, 6144)
Scaler guardado: C:\Users\USER\Documents\GitHub\ITEL\assets\models\final_mlp_scaler.joblib


In [15]:
NUM_MODELS  = 5
NUM_EPOCHS  = 50
BATCH_SIZE  = 32
LR          = 1e-3
NUM_CLASSES = len(le.classes_)

X_tr_t = torch.tensor(X_train_s, dtype=torch.float32)
y_tr_t = torch.tensor(y_train,   dtype=torch.long)
train_ds = TensorDataset(X_tr_t, y_tr_t)

base_models = []
for m_idx in range(NUM_MODELS):
    torch.manual_seed(ITELConfig.SEED + m_idx)
    model     = MLP(in_dim=ITELConfig.EMB_DIM, num_classes=NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    loader    = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    model.train()
    for epoch in range(NUM_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    model.eval()
    base_models.append(model)
    mp = ITELConfig.MODELS_DIR / f'base_mlp_{m_idx+1}_v5.pth'
    torch.save(model.state_dict(), mp)
    print(f'  Modelo {m_idx+1}/{NUM_MODELS} guardado.')
print('5 modelos base entrenados.')


  Modelo 1/5 guardado.
  Modelo 2/5 guardado.
  Modelo 3/5 guardado.
  Modelo 4/5 guardado.
  Modelo 5/5 guardado.
5 modelos base entrenados.


In [16]:
X_m_t = torch.tensor(X_meta_s, dtype=torch.float32).to(device)
probs_list = []
with torch.inference_mode():
    for m in base_models:
        logits = m(X_m_t)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        probs_list.append(probs)
X_stack = np.hstack(probs_list)
print(f'Stacking matrix: {X_stack.shape}  (esperado {len(y_meta)} x {NUM_MODELS*NUM_CLASSES})')


Stacking matrix: (373, 10)  (esperado 373 x 10)


In [17]:
meta_model = LogisticRegression(max_iter=1000, random_state=ITELConfig.SEED, C=1.0)
meta_model.fit(X_stack, y_meta)
print('Meta-set:')
print(classification_report(y_meta, meta_model.predict(X_stack), target_names=le.classes_))
meta_path = ITELConfig.MODELS_DIR / 'final_meta_model_v5.joblib'
joblib.dump(meta_model, meta_path)
print(f'Meta-modelo guardado: {meta_path}')


Meta-set:
              precision    recall  f1-score   support

     Healthy       0.92      0.90      0.91       168
    Patients       0.92      0.93      0.93       205

    accuracy                           0.92       373
   macro avg       0.92      0.92      0.92       373
weighted avg       0.92      0.92      0.92       373

Meta-modelo guardado: C:\Users\USER\Documents\GitHub\ITEL\assets\models\final_meta_model_v5.joblib


In [18]:
X_te_t = torch.tensor(X_test_s, dtype=torch.float32).to(device)
probs_test = []
with torch.inference_mode():
    for m in base_models:
        logits = m(X_te_t)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        probs_test.append(probs)
X_stack_test = np.hstack(probs_test)
y_pred_test  = meta_model.predict(X_stack_test)
print('TEST SET:')
print(classification_report(y_test, y_pred_test, target_names=le.classes_))


TEST SET:
              precision    recall  f1-score   support

     Healthy       0.93      0.88      0.90       168
    Patients       0.91      0.94      0.92       205

    accuracy                           0.91       373
   macro avg       0.92      0.91      0.91       373
weighted avg       0.91      0.91      0.91       373



In [19]:
le_path = ITELConfig.MODELS_DIR / 'label_encoder_v5.joblib'
joblib.dump(le, le_path)
print(f'LabelEncoder guardado: {le_path}')
print('\nArtefactos:')
arts = (
    [ITELConfig.MODELS_DIR / 'final_mlp_scaler.joblib',
     ITELConfig.MODELS_DIR / 'label_encoder_v5.joblib',
     ITELConfig.MODELS_DIR / 'final_meta_model_v5.joblib']
    + [ITELConfig.MODELS_DIR / f'base_mlp_{i+1}_v5.pth' for i in range(5)]
)
for a in arts:
    ok  = 'OK   ' if a.exists() else 'FALTA'
    sz  = f'{a.stat().st_size/1024:.0f} KB' if a.exists() else ''
    print(f'  [{ok}] {a.name}  {sz}')


LabelEncoder guardado: C:\Users\USER\Documents\GitHub\ITEL\assets\models\label_encoder_v5.joblib

Artefactos:
  [OK   ] final_mlp_scaler.joblib  145 KB
  [OK   ] label_encoder_v5.joblib  0 KB
  [OK   ] final_meta_model_v5.joblib  1 KB
  [OK   ] base_mlp_1_v5.pth  12954 KB
  [OK   ] base_mlp_2_v5.pth  12954 KB
  [OK   ] base_mlp_3_v5.pth  12954 KB
  [OK   ] base_mlp_4_v5.pth  12954 KB
  [OK   ] base_mlp_5_v5.pth  12954 KB
